In [0]:
from pyspark.sql.utils import AnalysisException
from datetime import date
from pyspark.sql.functions import col, from_unixtime

table_name = "dbt_job_market.default.linkedin_data"
today_str = date.today().strftime("%Y-%m-%d")

# 1. Create table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        id BIGINT GENERATED ALWAYS AS IDENTITY,
        job_id STRING,
        job_website STRING,
        search_keyword STRING,
        job_url STRING,
        job_title STRING,
        company_name STRING,
        company_url STRING,
        time_posted STRING,
        num_applicants STRING,
        location STRING,
        seniority_level STRING,
        employment_type STRING,
        job_function STRING,
        industries STRING,
        job_description STRING,
        scraped_at TIMESTAMP
    )
""")

try:
    is_empty = spark.sql(f"SELECT 1 FROM {table_name} LIMIT 1").count() == 0

    if is_empty:
        print("Table is empty → loading ALL files")
        df = spark.read.format("json") \
            .option("header", "true") \
            .load("/Volumes/dbt_job_market/default/linkedin_data_volume/*.json")
    else:
        print("Table not empty → loading TODAY files only")
        df = spark.read.format("json") \
            .option("header", "true") \
            .load(f"/Volumes/dbt_job_market/default/linkedin_data_volume/{today_str}*.json")

except AnalysisException:
    print("Table not found → loading ALL files")
    df = spark.read.format("json") \
        .option("header", "true") \
        .load("/Volumes/dbt_job_market/default/linkedin_data_volume/*.json")

# Transform scraped_at from number to timestamp
df = df.withColumn("scraped_at", from_unixtime(col("scraped_at")).cast("timestamp"))

df.write.mode("append").saveAsTable(table_name)

In [0]:
%sql
with full_table as (
  select distinct sl.*, sf.job_function as each_job_function, si.industry as each_industry from dbt_job_market.silver.silver_linkedin_data sl
  left join dbt_job_market.silver.silver_job_functions sf
  on sl.job_id = sf.job_id
  left join dbt_job_market.silver.silver_job_industries si
  on sl.job_id = si.job_id
  where sl.job_id = 'li-4371281442'
  -- where id = 10649
  -- where scraped_at is null
)
select * from full_table

In [0]:
%sql
-- DROP TABLE IF EXISTS dbt_job_market.bronze.bronze_linkedin_data

In [0]:
%sql
select * from dbt_job_market.silver.silver_linkedin_data

In [0]:
%sql
select job_id, job_function, industries from dbt_job_market.gold.gold_obt
-- select count(distinct company_name), count(distinct company_url), count(distinct industries) from dbt_job_market.gold.gold_obt
where company_name == 'Shopee'
order by job_id


In [0]:
display(spark.sql("DESCRIBE TABLE dbt_job_market.gold.gold_obt"))

In [0]:
%sql
select scraped_at, count(*) as rows from dbt_job_market.default.linkedin_data
group by scraped_at
order by scraped_at


In [0]:
%sql
select scraped_at, count(*) as rows from dbt_job_market.silver.silver_linkedin_data
group by scraped_at
order by scraped_at

In [0]:
%sql
select * from dbt_job_market.gold.dim_jobs
where job_id == 'li-4371281442'

In [0]:
%sql
select count(*) from dbt_job_market.gold.snap_job_applicants
-- where job_id = 'li-4262353945' and search_keyword = 'Data Engineer'
-- where dbt_scd_id = '8bd51e0ff17c5b381fb5f20726d38861'


In [0]:
%sql
-- select count(distinct *) from dbt_job_market.gold.dim_jobs
select count(*) from dbt_job_market.gold.dim_jobs
-- select * from dbt_job_market.gold.dim_jobs
-- select count(*) from dbt_job_market.silver.silver_linkedin_data

In [0]:
%sql
SELECT *
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY (job_id, search_keyword)
            ORDER BY scraped_at DESC
        ) AS rn
    FROM dbt_job_market.silver.silver_linkedin_data
)
-- WHERE rn = 1
where job_id = 'li-4262353945' and search_keyword = 'Data Engineer'

In [0]:
%sql
select * from dbt_job_market.gold.snap_job_applicants
